# Electrodynamics from Scratch — Lecture 7: The Multipole Expansion

Far away from any localized charge distribution, the details of where the charge sits stop mattering and only a few integrated moments survive. The reason is that the Coulomb kernel can be Taylor-expanded: with the source at the origin and the field point $\mathbf{r}$ much larger than the source size $a$, the potential is
$$
V(\mathbf{r}) = \frac{1}{4\pi\varepsilon_0}\int \frac{\rho(\mathbf{r}')\, d^3 r'}{|\mathbf{r} - \mathbf{r}'|}
= \frac{1}{4\pi\varepsilon_0}\left[ \frac{Q}{r} + \frac{\mathbf{p}\cdot\hat{\mathbf{r}}}{r^2} + \frac{1}{2}\sum_{i,j} Q_{ij}\, \hat{r}_i \hat{r}_j \over r^3 + \cdots \right],
$$
where the three leading moments are the **monopole** (total charge) $Q = \int \rho\, d^3r'$, the **dipole moment** $\mathbf{p} = \int \mathbf{r}'\, \rho\, d^3r'$, and the **quadrupole tensor** $Q_{ij} = \int (3 x'_i x'_j - r'^2\, \delta_{ij})\, \rho\, d^3r'$, which is traceless by construction. Each successive term is smaller by one more power of $a/r$, so at large $r$ the first non-vanishing moment dominates the physics.

This hierarchy is why a neutral molecule still polarizes its surroundings: with $Q = 0$ the $1/r$ term dies and the $\mathbf{p}\cdot\hat{\mathbf{r}}/r^2$ term takes over, and if the geometry is symmetric enough that $\mathbf{p}$ also vanishes, the traceless $Q_{ij}$ leads at order $1/r^3$. The fields decay one power faster than the potentials, so a dipole field falls as $1/r^3$ and a quadrupole field as $1/r^4$ — but the *potentials* fall as $1/r^2$ and $1/r^3$, and those are the cleanest power laws to measure. In this notebook we compute the moments of a few charge configurations with `eds.multipole_moments`, watch the far-field expansion converge onto the exact potential as we add terms, verify the $1/r^2$ versus $1/r^3$ decay by fitting power laws, and draw the field lines and equipotentials of a pure dipole and a linear quadrupole.

In [ ]:
%matplotlib inline
import numpy as np
import electrodynamics_from_scratch as eds

## Multipole moments of three charge configurations

Three configurations, three different leading moments: a single charge leads with its monopole, a neutral pair leads with its dipole (zero net charge, nonzero $\mathbf{p}$), and a linear quadrupole — two charges $+q$ symmetric about a charge $-2q$ — kills the dipole by symmetry and leads with the traceless tensor $Q_{ij}$. `eds.multipole_moments` computes $Q$, $\mathbf{p}$ and $Q_{ij}$ by direct integration over the point charges.

In [ ]:
q = 1.0
d = 0.2

configs = {
    "single charge": ([q],
                      np.array([[0.0, 0.0, 0.0]])),
    "neutral pair": ([-q, q],
                     np.array([[-d / 2, 0.0, 0.0], [d / 2, 0.0, 0.0]])),
    "linear quadrupole": ([q, q, -2 * q],
                          np.array([[0.0, 0.0, d], [0.0, 0.0, -d], [0.0, 0.0, 0.0]])),
}

for name, (charges, positions) in configs.items():
    m = eds.multipole_moments(charges, positions)
    print(name)
    print(f"  Q (monopole)  = {m['Q']:.6f}")
    print(f"  p (dipole)    = {np.round(m['p'], 6)}")
    print(f"  trace(Q)      = {np.trace(m['Q_tensor']):.2e}   (quadrupole tensor, should be 0)")
    print(f"  Q_tensor      =\n{np.round(m['Q_tensor'], 6)}")
    print()

## Far-field expansion versus the exact potential

For the neutral pair, the monopole term vanishes, so truncating the expansion at order 1 keeps only the dipole term ($\propto 1/r^2$) and truncating at order 2 adds the quadrupole term ($\propto 1/r^3$). Along the $x$-axis, `eds.exact_potential` sums the two Coulomb terms exactly while `eds.far_field_potential` uses the truncated series; `eds.expansion_error_vs_r` measures the relative error of each truncation as a function of distance. The quadrupole correction should buy roughly one extra power of $1/r$.

In [ ]:
charges = [-q, q]
positions = np.array([[0.0, 0.0, 0.0], [d, 0.0, 0.0]])

r_values = np.geomspace(0.5, 50.0, 12)
pts = np.stack([r_values, np.zeros_like(r_values), np.zeros_like(r_values)], axis=1)

V_exact = eds.exact_potential(charges, positions, pts)
V_order1 = eds.far_field_potential(charges, positions, pts, order=1)
V_order2 = eds.far_field_potential(charges, positions, pts, order=2)

i = 3
print(f"at r = {r_values[i]:.2f}:  exact = {V_exact[i]:.6f},  "
      f"order 1 = {V_order1[i]:.6f},  order 2 = {V_order2[i]:.6f}")

err_order1 = eds.expansion_error_vs_r(charges, positions, r_values, order=1)
err_order2 = eds.expansion_error_vs_r(charges, positions, r_values, order=2)

for label, err in [("order 1 (dipole only)", err_order1),
                   ("order 2 (+ quadrupole)", err_order2)]:
    _, exponent = eds.power_law_fit(r_values, err)
    print(f"{label}: error falls as r^{exponent:.2f}")

fig, ax = eds.plot_decay(
    r_values,
    {"order 1 (dipole only)": err_order1,
     "order 2 (+ quadrupole)": err_order2},
    title="Far-field expansion error for a neutral pair",
)
ax.set_xlabel("r / (source size)")
ax.set_ylabel("relative error")
fig

## Power-law decay: dipole versus quadrupole

Each term of the multipole expansion has its own signature decay in $r$: the dipole potential goes as $\mathbf{p}\cdot\hat{\mathbf{r}}/(4\pi\varepsilon_0 r^2)$ and the quadrupole potential as $\hat{\mathbf{r}}^\mathsf{T} Q \hat{\mathbf{r}}/(8\pi\varepsilon_0 r^3)$. Along the symmetry axis of each source we isolate one term at a time — `eds.point_dipole_potential` for a point dipole, `eds.quadrupole_potential` with the tensor computed from the linear quadrupole — and fit a power law with `eds.power_law_fit`. On the log–log plot each is a straight line whose slope is the negative power of $r$.

In [ ]:
p = np.array([0.0, 0.0, 1.0])
r0 = np.zeros(3)
q_tensor_quad = eds.multipole_moments(
    [q, q, -2 * q],
    np.array([[0.0, 0.0, d], [0.0, 0.0, -d], [0.0, 0.0, 0.0]]),
)["Q_tensor"]

r_values = np.geomspace(1.0, 100.0, 25)
pts = np.stack([np.zeros_like(r_values), np.zeros_like(r_values), r_values], axis=1)

V_dip = np.abs(eds.point_dipole_potential(p, r0, pts))
V_quad = np.abs(eds.quadrupole_potential(q_tensor_quad, r0, pts))

_, exp_dip = eds.power_law_fit(r_values, V_dip)
_, exp_quad = eds.power_law_fit(r_values, V_quad)
print(f"dipole term:     fitted exponent {exp_dip:.4f}   (expected -2)")
print(f"quadrupole term: fitted exponent {exp_quad:.4f}   (expected -3)")

fig, ax = eds.plot_decay(
    r_values,
    {"dipole term $|V|$ (expect $1/r^2$)": V_dip,
     "quadrupole term $|V|$ (expect $1/r^3$)": V_quad},
    title="Power-law decay of the multipole terms",
)
ax.set_xlabel("r")
ax.set_ylabel("|V|")
fig

## Field lines of a pure dipole

The field of a point dipole, $\mathbf{E} = (1/4\pi\varepsilon_0 r^3)\,[\,3(\mathbf{p}\cdot\hat{\mathbf{r}})\hat{\mathbf{r}} - \mathbf{p}\,]$, is exactly what the far-field expansion predicts a tiny neutral pair looks like from outside. In the plane containing $\mathbf{p}$ the field lines leave the positive end, loop around, and return to the negative end. We build the 2D field from `eds.point_dipole_field` and mask the singular origin before handing the function to `eds.plot_field_lines`.

In [ ]:
p = np.array([0.0, 0.0, 1.0])
r0 = np.zeros(3)

def dipole_field_2d(X, Y):
    near_origin = (X**2 + Y**2) < 0.05**2   # the field diverges at the dipole
    pts = np.stack([X[~near_origin].ravel(), Y[~near_origin].ravel(),
                    np.zeros(X[~near_origin].size)], axis=1)
    E = eds.point_dipole_field(p, r0, pts)
    Ex = np.full(X.shape, np.nan)
    Ey = np.full(X.shape, np.nan)
    Ex[~near_origin] = E[:, 0]
    Ey[~near_origin] = E[:, 1]
    return Ex, Ey

fig, ax = eds.plot_field_lines(dipole_field_2d, extent=2.0,
                               title="Field lines of a pure dipole ($p$ along $z$)")
fig

## Equipotentials of a linear quadrupole

For the linear quadrupole the dipole moment vanishes identically, so far from the source the potential is governed by the traceless tensor $Q_{ij}$: positive lobes along the $z$-axis where $\hat{\mathbf{r}}^\mathsf{T} Q \hat{\mathbf{r}} > 0$ and negative lobes in the equatorial plane. We take $Q_{ij}$ from `eds.multipole_moments`, evaluate `eds.quadrupole_potential` on a slice through the charges, and let `eds.plot_equipotentials` draw the contours — near the origin the equipotentials crowd together, reflecting the $1/r^3$ singularity.

In [ ]:
q4_charges = [q, q, -2 * q]
q4_positions = np.array([[0.0, 0.0, d], [0.0, 0.0, -d], [0.0, 0.0, 0.0]])
q_tensor_quad = eds.multipole_moments(q4_charges, q4_positions)["Q_tensor"]

def quadrupole_potential_2d(X, Y):
    # the potential diverges at the quadrupole centre and at the charges
    near = ((X**2 + (Y - d) ** 2) < 0.15**2) | \
           ((X**2 + (Y + d) ** 2) < 0.15**2) | \
           ((X**2 + Y**2) < 0.15**2)
    pts = np.stack([X[~near].ravel(), Y[~near].ravel(),
                    np.zeros(X[~near].size)], axis=1)
    V = eds.quadrupole_potential(q_tensor_quad, np.zeros(3), pts)
    Vg = np.full(X.shape, np.nan)
    Vg[~near] = V
    return Vg

fig, ax = eds.plot_equipotentials(quadrupole_potential_2d, extent=2.0,
                                  title="Equipotentials of a linear quadrupole")
fig

## Exercises

1. **Conceptual — spherical symmetry.** A spherically symmetric charge distribution has $Q \neq 0$ but every higher moment vanishes. Show from the definitions of $\mathbf{p}$ and $Q_{ij}$ why symmetry kills them, and conclude that outside such a distribution the field is exactly the point-charge field (compare with `eds.shell_field` from the package).

2. **Coding — a tetrahedron of charge.** Place four equal charges $+q$ at the vertices of a regular tetrahedron centered on the origin (e.g. at $(1,1,1)$, $(1,-1,-1)$, $(-1,1,-1)$, $(-1,-1,1)$, suitably scaled). Use `eds.multipole_moments` to verify that the dipole moment vanishes and that $Q_{ij}$ is traceless, then use `eds.exact_potential` and `eds.far_field_potential` to check that the order-1 far-field error is far smaller than for the neutral pair of this notebook.

3. **Coding — translating the source.** Shift the neutral pair of this notebook so both charges sit at $\mathbf{a} = (0.5, -0.3, 0.2)$ plus their original offsets, recompute `eds.multipole_moments`, and explain how $Q$, $\mathbf{p}$ and $Q_{ij}$ change. Which moments are invariant under a translation of the whole configuration, and why?

4. **Coding — the charged case.** For a configuration with $Q \neq 0$ (say a single charge, or the neutral pair plus one extra charge at the origin), use `eds.expansion_error_vs_r` with `order=0` and `eds.power_law_fit` to confirm that the monopole truncation error falls as $1/r^2$. Then compare against the neutral case: how much farther away must you stand for the same accuracy when the monopole is present?

5. **Coding — the dipole field.** Along the axis of the point dipole, the field magnitude is $|\mathbf{E}| = 2|\mathbf{p}|/(4\pi\varepsilon_0 r^3)$. Use `eds.point_dipole_field` on a ray of points along the $z$-axis and `eds.power_law_fit` on $|\mathbf{E}|$ to verify the $1/r^3$ decay of the dipole *field* (one power faster than the dipole potential of this notebook), and check the prefactor against the analytic expression using `eds.EPSILON_0`.

6. **Conceptual — when is the far field good enough?** Using the error curves from the neutral-pair cell, estimate the distance (in units of the charge separation $d$) at which the order-1 expansion is accurate to 1%. Then argue physically why adding the quadrupole term matters less at large $r$ but a lot at $r \sim 5d$.